In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

df.display() 

In [0]:
df=df.withColumn("amount",col("amount").cast("string"))
df.write.format("delta")\
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/deltalogs")


In [0]:
df=spark.read.format("delta") \
    .load("/Volumes/workspace/default/deltalogs")
df.display()

In [0]:
new_data=[(5, "C005", "Camera", 30000)]
df_new=spark.createDataFrame(new_data,columns)



In [0]:
from pyspark.sql.functions import *

df_fixed=df_new.withColumn("amount",col("amount").cast("string"))

df_fixed.write.format("delta")\
    .mode("append") \
    .option("mergeSchema", 'true') \
    .save("/Volumes/workspace/default/deltalogs")

In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark,"/Volumes/workspace/default/deltalogs") 

delta_table.update(
    condition = "id = 2 ",
    set = {"amount" : "18000"}
)

In [0]:
updates = [
    (3, "C003", "Tablet", "22000"),  # update
    (6, "C006", "Watch", "8000")     # insert
]
updated_df=spark.createDataFrame(updates,columns)

updated_df.write.format("delta") \
                .mode("append") \
                .option("mergeSchema", 'true') \
                .save("/Volumes/workspace/default/deltalogs")

In [0]:
updated_df=spark.read.format("delta") \
                .load("/Volumes/workspace/default/deltalogs")
updated_df.display()

In [0]:
delta_table.alias("target").merge(
    updated_df.alias("source"),
    "target.id = source.id"
).whenNotMatchedInsert(values={
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"}).execute()

In [0]:
new_data1 = [(6, "F", 10000, "India")]
new_columns1 = ["id", "name", "salary", "country"]

In [0]:
df_new1=spark.createDataFrame(new_data1,new_columns1)
df_new1.write.format("delta") \
                .mode("append") \
                .option("mergeSchema","true") \
                .save("/Volumes/workspace/default/deltalogs")

In [0]:
df_new1=spark.read.format("delta") \
                    .load("/Volumes/workspace/default/deltalogs")
df_new1.display()
delta_table.alias("target").merge(
    df_new1.alias("source"),
    "target.id = source.id"
).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary",
    "country": "source.country"}).execute()

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/deltalogs")
df_old.display()

In [0]:
delta_table.restoreToVersion(0)